# Cyberbullying Detector

This notebook walks through a beginner-friendly machine learning workflow for spotting cyber-aggressive language in social media text.

The goal is not to decide a person's character from one post. Instead, we build a model that can flag harmful language for review, while keeping human judgment in the loop.

We will:
- load the dataset
- clean the tweet text
- remove invalid or conflicting labels
- train a TF-IDF + logistic regression model
- evaluate the results and save the model


In [ ]:
# Step 1: check that the dataset file is present in the project folder.
from pathlib import Path

dataset_path = Path("../data/Dataset_for_Detection_of_Cyber-Trolls_1.json")
print(f"Dataset exists: {dataset_path.exists()}")
print(f"Dataset path: {dataset_path}")


In [ ]:
# Step 2: load the raw dataset and inspect its structure.
# The source file is JSON Lines, where each line is one tweet and its annotation.
import pandas as pd

raw = pd.read_json(dataset_path, lines=True)
print(f"Raw rows: {len(raw):,}")
print(raw.head(3).to_string())


In [ ]:
# Step 3: extract the text and the label from each record.
# The label is nested inside annotation.label, so we need to unwrap it carefully.
import html

def first_label(annotation):
    """Return the first label found in the nested annotation object."""
    if not isinstance(annotation, dict):
        return None
    labels = annotation.get("label")
    if not isinstance(labels, list) or not labels:
        return None
    return str(labels[0]).strip()

data = pd.DataFrame({
    "text": raw.get("content"),
    "label": raw.get("annotation", pd.Series(index=raw.index, dtype=object)).map(first_label),
})

# Clean the text by removing extra whitespace and HTML symbols like &amp;.
data["text"] = data["text"].fillna("").astype(str).map(
    lambda text: " ".join(html.unescape(text).split())
)
data["label"] = data["label"].astype("string").str.strip()

# Keep only rows with valid labels (0 = non aggressive, 1 = cyber-aggressive).
data = data[data["text"].str.strip().ne("") & data["label"].isin(["0", "1"])].copy()
data["label"] = data["label"].astype(int)

print(f"Valid rows after cleaning: {len(data):,}")
print(data.head(5).to_string(index=False))


In [ ]:
# Step 4: remove contradictory labels before training.
# Some repeated messages can have different labels, which would confuse the model.
data["text_key"] = data["text"].str.casefold()
conflicting_text = data.groupby("text_key")["label"].transform("nunique").gt(1)
removed_conflicts = data.loc[conflicting_text, "text_key"].nunique()
data = data.loc[~conflicting_text].drop_duplicates("text_key", keep="first")

print(f"Conflicting repeated texts removed: {removed_conflicts:,}")
print(f"Rows remaining for training: {len(data):,}")
print(data["label"].value_counts().sort_index().to_string())


In [ ]:
# Step 5: split the data into training and test sets.
# We want a fair test set that keeps the class balance close to the original dataset.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data["text"],
    data["label"],
    test_size=0.20,
    random_state=42,
    stratify=data["label"],
)

print(f"Training examples: {len(X_train):,}")
print(f"Testing examples: {len(X_test):,}")


In [ ]:
# Step 6: build the model pipeline.
# TF-IDF turns words into numeric features, and logistic regression learns the pattern.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_features=30_000,
            sublinear_tf=True,
        ),
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1_200,
            class_weight="balanced",
            random_state=42,
        ),
    ),
])

model.fit(X_train, y_train)
print("Model trained successfully.")


In [ ]:
# Step 7: evaluate how well the model performs on unseen messages.
from sklearn.metrics import classification_report, confusion_matrix

predictions = model.predict(X_test)
print(classification_report(
    y_test,
    predictions,
    labels=[0, 1],
    target_names=["non-aggressive", "cyber-aggressive"],
    zero_division=0,
))
print("Confusion matrix:")
print(confusion_matrix(y_test, predictions, labels=[0, 1]))


In [ ]:
# Step 8: save the trained pipeline so it can be reused later on new text.
import joblib

joblib.dump(model, "../cyberbully_baseline.joblib")
print("Model saved to ../cyberbully_baseline.joblib")

# Example prediction to show the system in action.
sample_text = ["You are so pathetic and everyone hates you."]
prediction = model.predict(sample_text)[0]
probability = model.predict_proba(sample_text)[0][prediction]
print(f"Sample text label: {prediction}")
print(f"Sample text confidence: {probability:.2f}")


## Final note

This project is a helpful research tool for detecting harmful language, not a final judge of a person's character. It should always be paired with human review and context.
